# Load Data

In [ ]:
import pandas as pd
import os

# Local path to save the data
local_path = "../data/imdb_sample.parquet"

# Check if the file already exists locally
if os.path.exists(local_path):
    print("Loading from local file...")
    df = pd.read_parquet(local_path)
else:
    print("Downloading and saving locally...")
    splits = {
        "train": "plain_text/train-00000-of-00001.parquet",
        "test": "plain_text/test-00000-of-00001.parquet",
        "unsupervised": "plain_text/unsupervised-00000-of-00001.parquet",
    }
    # Download sample and save
    df = pd.read_parquet("hf://datasets/stanfordnlp/imdb/" + splits["train"]).sample(
        200
    )
    os.makedirs("../data", exist_ok=True)
    df.to_parquet(local_path)

Loading from local file...


In [28]:
df

,text,label
6868,"Dumb is as dumb does, in this thoroughly unint...",0
24016,I dug out from my garage some old musicals and...,1
9668,After watching this movie I was honestly disap...,0
13640,This movie was nominated for best picture but ...,1
14018,Just like Al Gore shook us up with his painful...,1
...,...,...
1747,"OK, I really don't have too much to say about ...",0
323,This movie is by far the worst movie ever made...,0
10296,SPOILER ALERT In this generic and forgettable ...,0
12750,"A few years ago, a friend got from one of his ...",1


# Setup for Transformers

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

from sklearn.model_selection import train_test_split

from datasets import Dataset

import torch

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Split into train/test sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"], df["label"], test_size=0.2
)


def tokenize_dataset(tokenizer):
    train_encodings = tokenizer(train_texts.tolist(), truncation=True, padding=True)
    val_encodings = tokenizer(val_texts.tolist(), truncation=True, padding=True)
    return train_encodings, val_encodings

# Fine-tuning Strategies

A. Freeze Base Model Weights

In [31]:
def freeze_model(model):
    for param in model.base_model.parameters():
        param.requires_grad = False

B. Reconstruct/Unfreeze Top Layers (partial fine-tuning)


In [ ]:
def unfreeze_last_layers(model, n_layers=2):
    # Freeze all
    for param in model.parameters():
        param.requires_grad = False
    # Unfreeze classifier
    for name, param in model.named_parameters():
        if "classifier" in name:
            param.requires_grad = True
    # Unfreeze last n encoder layers
    for name, param in model.named_parameters():
        for i in range(12 - n_layers, 12):  # 12 is typical for base models
            if f"layer.{i}" in name or f"transformer.layer.{i}" in name:
                param.requires_grad = True

# Training & Evaluation Function

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

from datasets import Dataset

from sklearn.metrics import accuracy_score

import torch


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = torch.tensor(logits).argmax(dim=1)

    return {"accuracy": accuracy_score(labels, predictions)}



def train_distilbert(strategy):

    return train_with_model("distilbert-base-uncased", strategy)



def train_bert(strategy):

    return train_with_model("bert-base-uncased", strategy)



def train_with_model(model_name, strategy):

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2
    ).to(device)


    train_encodings, val_encodings = tokenize_dataset(tokenizer)


    train_dataset = Dataset.from_dict(
        {

            "input_ids": train_encodings["input_ids"],

            "attention_mask": train_encodings["attention_mask"],

            "labels": train_labels.tolist(),

        }
    )

    val_dataset = Dataset.from_dict(
        {

            "input_ids": val_encodings["input_ids"],

            "attention_mask": val_encodings["attention_mask"],

            "labels": val_labels.tolist(),

        }
    )


    if strategy == "freeze":

        freeze_model(model)

    elif strategy == "reconstruct":

        unfreeze_last_layers(model)


    training_args = TrainingArguments(

        output_dir="./results",

        per_device_train_batch_size=32,

        per_device_eval_batch_size=32,

        num_train_epochs=2,

        logging_dir="./logs",

        logging_steps=10,
    )


    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,

        eval_dataset=val_dataset,

        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )


    trainer.train()

    eval_results = trainer.evaluate()

    return eval_results["eval_accuracy"]

# Comparison

In [ ]:
models = {
    "minilm-l6-h384-uncased": lambda strategy: train_with_model(
        "nreimers/MiniLM-L6-H384-uncased", strategy
    ),
    "distilbert-base-uncased": train_distilbert,
}
strategies = ["freeze", "reconstruct"]
results = {}

for model_name, train_fn in models.items():
    for strategy in strategies:
        print(f"Training {model_name} with strategy: {strategy}")
        accuracy = train_fn(strategy)
        results[f"{model_name}_{strategy}"] = accuracy

print("\nComparison Results:")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

Training minilm-l6-h384-uncased with strategy: freeze


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nreimers/MiniLM-L6-H384-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\yan4etooo\AppData\Local\Temp\ipykernel_19784\2765246430.py:52: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
10,0.692500


Training minilm-l6-h384-uncased with strategy: reconstruct


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nreimers/MiniLM-L6-H384-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\yan4etooo\AppData\Local\Temp\ipykernel_19784\2765246430.py:52: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


...

Step,Training Loss
10,0.690200


Training distilbert-base-uncased with strategy: reconstruct


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\yan4etooo\AppData\Local\Temp\ipykernel_19784\2765246430.py:52: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
10,0.683400



Comparison Results:
minilm-l6-h384-uncased_freeze: 0.4750
minilm-l6-h384-uncased_reconstruct: 0.4750
distilbert-base-uncased_freeze: 0.5500
distilbert-base-uncased_reconstruct: 0.4750
